# WanderWise+ Complete End-to-End Testing Notebook

**Intelligent Multi-Day Tourism Route Planning for Goa, India**

This notebook demonstrates the complete WanderWise+ workflow:
1. **Module 1**: User preference filtering (NLC simulation)
2. **Module 2**: Popularity scoring & normalization (WPI)
3. **Module 3**: Geographic clustering (K-Means by day)
4. **Module 4**: Route optimization (Genetic Algorithm)

---

## Architecture Overview

```
User Natural Language Input
        ↓
Module 1: Interest Classification (8 categories)
        ↓
Filter POIs by preferences + min rating
        ↓
Module 2: Calculate WPI (Weighted Popularity Index)
        ↓
Module 3: K-Means Clustering (K = num_days)
        ↓
Module 4: GA Route Optimization (per day)
        ↓
Interactive Maps + Timeline + Export JSON
```

**Algorithm:** TPOS-aligned Genetic Algorithm  
**Reference:** *Travel Planning Optimization System Employing Genetic Algorithms*  
Rusu & Alexandrescu, ICSTCC 2024

---

## System Requirements

### Python Packages
```bash
pip install pandas numpy scikit-learn folium plotly requests
```

### Optional: OSRM Server (for real routing)
**Docker Setup (Recommended):**
```bash
# Download India OSM data
wget http://download.geofabrik.de/asia/india-latest.osm.pbf

# Extract and prepare
docker run -t -v "${PWD}:/data" ghcr.io/project-osrm/osrm-backend osrm-extract -p /opt/car.lua /data/india-latest.osm.pbf
docker run -t -v "${PWD}:/data" ghcr.io/project-osrm/osrm-backend osrm-partition /data/india-latest.osrm
docker run -t -v "${PWD}:/data" ghcr.io/project-osrm/osrm-backend osrm-customize /data/india-latest.osrm

# Run OSRM server
docker run -t -i -p 5000:5000 -v "${PWD}:/data" ghcr.io/project-osrm/osrm-backend osrm-routed --algorithm mld /data/india-latest.osrm
```

**Fallback:** If OSRM unavailable, notebook uses haversine distance + speed-based time estimation.

---

## Data Files Required
- `Wanderwise_datasetnew.csv` (100+ Goa POIs with ratings, coordinates, categories)
- Alternative: `goa_tourist_attractions.csv`

## 1. Setup & Configuration

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import random
import json
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from datetime import datetime, timedelta
from math import radians, sin, cos, asin, sqrt
import warnings
warnings.filterwarnings('ignore')

# Clustering
from sklearn.cluster import KMeans

# Visualization
import folium
from folium.plugins import MarkerCluster

# Requests for OSRM
import requests

# Plotly for charts
try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("⚠️ Plotly not available. Install with: pip install plotly")

print("✅ All imports loaded successfully")
print(f"📊 Plotly available: {PLOTLY_AVAILABLE}")

### User Configuration Parameters

**Modify these cells to customize your trip:**

In [ ]:
# ========================================
# USER CONFIGURATION - MODIFY AS NEEDED
# ========================================

# Trip Configuration
NUM_DAYS = 3                    # Number of days for trip planning
MIN_RATING = 4.0                # Minimum POI rating threshold (0-5)

# User Preferences (Natural Language)
USER_PREFERENCE_INPUT = """
I love exploring historical forts and beautiful beaches. 
I'm interested in nature waterfalls and adventure activities.
Not interested in nightlife or shopping.
"""

# Route Configuration
MIN_POIS_PER_ROUTE = 4          # Minimum POIs per day
MAX_POIS_PER_ROUTE = 15         # Maximum POIs per day

print(f"🗓️  Trip Duration: {NUM_DAYS} days")
print(f"⭐ Minimum Rating: {MIN_RATING}")
print(f"📍 POIs per day: {MIN_POIS_PER_ROUTE}-{MAX_POIS_PER_ROUTE}")
print(f"\n💬 User Preferences:\n{USER_PREFERENCE_INPUT.strip()}")

In [ ]:
# ========================================
# GA PARAMETERS (TPOS-ALIGNED)
# ========================================
# Reference: Travel Planning Optimization System Employing Genetic Algorithms
# Rusu & Alexandrescu, ICSTCC 2024

# Population & Evolution
POPULATION_SIZE = 100           # Fixed initial population per day
MAX_GENERATIONS = 50            # Stops on fitness plateau or max iterations
CROSSOVER_RATE = 0.7            # Single-point crossover probability (TPOS spec)
MUTATION_RATE = 0.2             # Swap mutation probability (TPOS spec)
TOURNAMENT_SIZE = 2             # Two random individuals compete (TPOS spec)
ELITE_COUNT = 1                 # Best solution preserved (TPOS spec)
EARLY_STOPPING_THRESHOLD = 10   # Stop if no improvement for N generations

# Penalty Values (SCALED FOR GOA DATA)
# Note: Original TPOS multipliers calibrated for Paris data
# Adjusted for Goa's distance (0-50km) and popularity (0-1) ranges
HARD_VIOLATION_PENALTY = 1000   # Per gene: closed POI or exceeds open hours (TPOS: 10000)
USER_PREFERENCE_PENALTY_MULTIPLIER = 1.0   # (100 - preference%) × 1 (TPOS: 10)
DISTANCE_PENALTY_MULTIPLIER = 1.0          # walking_distance × 1 (TPOS: 10000)
MUST_SEE_PENALTY_MULTIPLIER = 100          # (β - α) × 100 (TPOS: 1000)
RESTAURANT_PENALTY = 100                   # Incorrect restaurant count/placement (TPOS: 1000)

print("⚙️  GA Configuration (TPOS Paper Aligned):")
print(f"   Population: {POPULATION_SIZE} chromosomes")
print(f"   Generations: {MAX_GENERATIONS} (with plateau detection)")
print(f"   Crossover: {CROSSOVER_RATE*100:.0f}% (single-point)")
print(f"   Mutation: {MUTATION_RATE*100:.0f}% (swap only)")
print(f"   Selection: Tournament (k={TOURNAMENT_SIZE})")
print(f"   Elitism: {ELITE_COUNT} best preserved")
print(f"\n⚠️  TPOS Penalty System (Scaled for Goa):")
print(f"   Hard violation: {HARD_VIOLATION_PENALTY} per gene")
print(f"   Distance: {DISTANCE_PENALTY_MULTIPLIER}× km")
print(f"   User pref: {USER_PREFERENCE_PENALTY_MULTIPLIER}× diff")
print(f"   Must-see: {MUST_SEE_PENALTY_MULTIPLIER}× gap")

In [ ]:
# ========================================
# TIME CONFIGURATION
# ========================================

# Tour Schedule
TOUR_START_TIME = "09:00"       # Day start time (TPOS spec)
TOUR_END_TIME = "16:00"         # Day end/trim time (TPOS spec ~04:00 PM)
LUNCH_START_TIME = "12:00"
LUNCH_END_TIME = "13:30"
DAILY_TIME_BUDGET_HOURS = 7     # 09:00 to 16:00 = 7 hours

# POI Default Values
DEFAULT_VISIT_DURATION_MIN = 60
DEFAULT_OPENING_TIME = "09:00"
DEFAULT_CLOSING_TIME = "18:00"

# Visit Duration by Location Type (TPOS Section 7)
VISIT_DURATIONS = {
    'museum': 120,
    'park': 60,
    'shopping_mall': 120,
    'zoo': 180,
    'fort': 90,
    'beach': 90,
    'waterfall': 60,
    'church': 45,
    'temple': 45,
    'default': 60
}

# Travel Speed (Goa road conditions)
AVERAGE_SPEED_KM_H = 30

print("⏰ Time Configuration:")
print(f"   Tour: {TOUR_START_TIME} - {TOUR_END_TIME} ({DAILY_TIME_BUDGET_HOURS}h)")
print(f"   Lunch: {LUNCH_START_TIME} - {LUNCH_END_TIME}")
print(f"   Average speed: {AVERAGE_SPEED_KM_H} km/h")
print(f"\n🏛️  Visit Durations:")
for loc_type, duration in sorted(VISIT_DURATIONS.items()):
    if loc_type != 'default':
        print(f"   {loc_type}: {duration} min")

In [ ]:
# ========================================
# OSRM CONFIGURATION
# ========================================

OSRM_BASE_URL = "http://localhost:5000"
USE_OSRM = True  # Set to False to force haversine fallback

# Test OSRM connectivity
def check_osrm_server():
    """Check if OSRM server is available."""
    if not USE_OSRM:
        return False
    try:
        response = requests.get(f"{OSRM_BASE_URL}/health", timeout=2)
        return response.status_code == 200
    except:
        return False

osrm_available = check_osrm_server()
print(f"🗺️  OSRM Server: {'✅ Available' if osrm_available else '❌ Not available (using haversine fallback)'}")
if not osrm_available:
    print("   To enable OSRM, see Docker setup instructions in markdown cells above")

In [ ]:
# Global state for distance/duration matrices
CURRENT_DISTANCE_KM = None
CURRENT_DURATION_MIN = None
CURRENT_POI_INDEX = {}

print("✅ Configuration complete!")

## 2. Data Loading & Validation

In [ ]:
# Load dataset
try:
    df = pd.read_csv('Wanderwise_datasetnew.csv')
    data_source = 'Wanderwise_datasetnew.csv'
except FileNotFoundError:
    try:
        df = pd.read_csv('goa_tourist_attractions.csv')
        data_source = 'goa_tourist_attractions.csv'
    except FileNotFoundError:
        raise FileNotFoundError("Neither Wanderwise_datasetnew.csv nor goa_tourist_attractions.csv found!")

print(f"📂 Loaded {len(df)} places from {data_source}")
print(f"\n📋 Columns: {list(df.columns)}")

In [ ]:
# Validate required columns
required_cols = ['name', 'latitude', 'longitude', 'rating']
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Check for missing coordinates or ratings
missing_coords = df[['latitude', 'longitude']].isnull().any(axis=1).sum()
missing_ratings = df['rating'].isnull().sum()

print(f"✅ All required columns present")
print(f"⚠️  Missing coordinates: {missing_coords} places")
print(f"⚠️  Missing ratings: {missing_ratings} places")

# Drop places with missing essential data
df = df.dropna(subset=['latitude', 'longitude', 'rating']).copy()
print(f"\n📍 {len(df)} places after removing incomplete data")

In [ ]:
# Display data statistics
print("📊 Dataset Statistics:")
print(f"   Total places: {len(df)}")
print(f"   Average rating: {df['rating'].mean():.2f}")
print(f"   Rating range: {df['rating'].min():.1f} - {df['rating'].max():.1f}")
print(f"\n🌍 Geographic Bounds (Goa):")
print(f"   Latitude: {df['latitude'].min():.4f} to {df['latitude'].max():.4f}")
print(f"   Longitude: {df['longitude'].min():.4f} to {df['longitude'].max():.4f}")

# Show top-rated places
print(f"\n⭐ Top 5 Rated Places:")
top_places = df.nlargest(5, 'rating')[['name', 'rating', 'user_ratings_total']]
for idx, row in top_places.iterrows():
    print(f"   {row['name']}: {row['rating']}⭐ ({row.get('user_ratings_total', 'N/A')} reviews)")

In [ ]:
# Parse categories from 'types' or 'categories' column
def parse_categories(row):
    """Extract category list from place data."""
    # Try 'categories' column first (if already parsed)
    if 'categories' in row and pd.notna(row['categories']):
        if isinstance(row['categories'], list):
            return row['categories']
        # Parse string representation of list
        try:
            import ast
            return ast.literal_eval(row['categories'])
        except:
            pass
    
    # Fall back to parsing 'types' or 'name' for keywords
    categories = []
    text = ''
    if 'types' in row and pd.notna(row['types']):
        text += str(row['types']).lower() + ' '
    text += str(row['name']).lower()
    
    # Map keywords to categories
    if any(word in text for word in ['fort', 'trek', 'adventure', 'safari', 'water sports']):
        categories.append('adventure')
    if 'beach' in text:
        categories.append('beaches')
    if any(word in text for word in ['food', 'restaurant', 'market', 'cuisine']):
        categories.append('food')
    if any(word in text for word in ['fort', 'museum', 'historical', 'heritage', 'palace']):
        categories.append('historical')
    if any(word in text for word in ['waterfall', 'park', 'wildlife', 'sanctuary', 'nature', 'garden']):
        categories.append('nature')
    if any(word in text for word in ['club', 'nightlife', 'bar', 'casino']):
        categories.append('nightlife')
    if any(word in text for word in ['temple', 'church', 'religious', 'basilica', 'cathedral']):
        categories.append('religious')
    if any(word in text for word in ['shopping', 'mall', 'market']):
        categories.append('shopping')
    
    return list(set(categories)) if categories else ['other']

# Apply category parsing
if 'categories' not in df.columns or df['categories'].isnull().all():
    df['categories'] = df.apply(parse_categories, axis=1)
else:
    # Ensure categories is list type
    df['categories'] = df.apply(parse_categories, axis=1)

# Count places by category
all_categories = {}
for cats in df['categories']:
    for cat in cats:
        all_categories[cat] = all_categories.get(cat, 0) + 1

print("\n🏷️  Places by Category:")
for cat, count in sorted(all_categories.items(), key=lambda x: x[1], reverse=True):
    print(f"   {cat:15s}: {count:3d} places ({count/len(df)*100:.1f}%)")

print(f"\n✅ Data loading and validation complete!")

## 3. Module 1: User Preference Filtering

Simulates NLC (Natural Language Classification) to extract user interests from natural language input.

In [ ]:
# Category keywords for classification
CATEGORY_KEYWORDS = {
    'adventure': ['fort', 'forts', 'trek', 'trekking', 'adventure', 'safari', 'water sports', 'kayaking', 'hiking'],
    'beaches': ['beach', 'beaches', 'coastal', 'seaside', 'shore', 'sunset'],
    'food': ['food', 'cuisine', 'restaurant', 'dining', 'eating', 'market', 'local dishes'],
    'historical': ['historical', 'history', 'heritage', 'ancient', 'monument', 'museum', 'fort', 'palace', 'architecture'],
    'nature': ['nature', 'waterfall', 'waterfalls', 'wildlife', 'sanctuary', 'park', 'garden', 'forest', 'natural'],
    'nightlife': ['nightlife', 'club', 'clubs', 'bar', 'bars', 'party', 'casino', 'entertainment'],
    'religious': ['religious', 'temple', 'temples', 'church', 'churches', 'spiritual', 'worship', 'basilica', 'cathedral'],
    'shopping': ['shopping', 'shop', 'shops', 'market', 'markets', 'mall', 'bazaar', 'souvenirs']
}

print("✅ Category keywords defined (8 categories)")

In [ ]:
def simulate_nlc_classification(user_input: str) -> Dict[str, bool]:
    """
    Simulate NLC model classification based on keyword matching.
    
    In production, this would call the hosted NLC model.
    Here we use simple keyword matching with sentiment detection.
    
    Args:
        user_input: Natural language preference text
    
    Returns:
        Dictionary with category: positive/negative classification
    """
    text = user_input.lower()
    
    # Sentiment indicators
    positive_words = ['love', 'enjoy', 'like', 'interested', 'want', 'prefer', 'looking for', 'exploring']
    negative_words = ['hate', 'dislike', 'not interested', 'avoid', 'boring', 'no ', "don't", 'not ']
    
    results = {}
    
    for category, keywords in CATEGORY_KEYWORDS.items():
        # Check if category is mentioned
        category_mentioned = any(keyword in text for keyword in keywords)
        
        if category_mentioned:
            # Determine sentiment (check if negative words appear near category keywords)
            is_negative = False
            for neg_word in negative_words:
                if neg_word in text:
                    # Check if category keyword appears after negative word within 50 chars
                    neg_pos = text.find(neg_word)
                    for keyword in keywords:
                        if keyword in text:
                            kw_pos = text.find(keyword)
                            if 0 <= kw_pos - neg_pos <= 50:
                                is_negative = True
                                break
            
            results[category] = not is_negative  # True = positive, False = negative
    
    return results

print("✅ NLC simulation function defined")

In [ ]:
# Classify user preferences
user_interests = simulate_nlc_classification(USER_PREFERENCE_INPUT)

positive_interests = [k for k, v in user_interests.items() if v]
negative_interests = [k for k, v in user_interests.items() if not v]

print(f"💬 User Input:\n{USER_PREFERENCE_INPUT.strip()}")
print(f"\n✅ Positive Interests: {positive_interests}")
print(f"❌ Negative Interests: {negative_interests}")

In [ ]:
def filter_places_by_interests(
    df: pd.DataFrame,
    positive_interests: List[str],
    negative_interests: List[str],
    min_rating: float = 4.0
) -> pd.DataFrame:
    """
    Filter places based on user interests.
    
    Args:
        df: DataFrame with places
        positive_interests: Categories user is interested in
        negative_interests: Categories user wants to avoid
        min_rating: Minimum rating threshold
    
    Returns:
        Filtered DataFrame
    """
    if not positive_interests:
        print("⚠️  No positive interests specified. Using all places.")
        filtered_df = df[df['rating'] >= min_rating].copy()
    else:
        # Filter: must have at least one positive interest
        def has_positive_interest(categories):
            return any(cat in positive_interests for cat in categories)
        
        # Filter: must not have negative interests
        def has_negative_interest(categories):
            if not negative_interests:
                return False
            return any(cat in negative_interests for cat in categories)
        
        filtered_df = df[
            df['categories'].apply(has_positive_interest) &
            ~df['categories'].apply(has_negative_interest) &
            (df['rating'] >= min_rating)
        ].copy()
    
    return filtered_df

# Apply filtering
filtered_df = filter_places_by_interests(df, positive_interests, negative_interests, MIN_RATING)

print(f"\n📍 Filtered Results:")
print(f"   Original places: {len(df)}")
print(f"   After filtering: {len(filtered_df)} places")
print(f"   Reduction: {(1 - len(filtered_df)/len(df))*100:.1f}%")

if len(filtered_df) == 0:
    raise ValueError("No places match your preferences! Try adjusting your input or lowering MIN_RATING.")

print(f"\n✅ Module 1 complete: {len(filtered_df)} places match your preferences")

## 4. Module 2: Popularity Scoring & Normalization (WPI)

Calculate Weighted Popularity Index (WPI) combining Google rating and review volume.

In [ ]:
# Ensure user_ratings_total column exists
if 'user_ratings_total' not in filtered_df.columns:
    filtered_df['user_ratings_total'] = 100  # Default value
    print("⚠️  user_ratings_total column missing, using default value 100")

# Step 1: Normalize Google rating (0-5 scale) to 0-1
filtered_df['normalized_google_rating'] = filtered_df['rating'] / 5.0

# Step 2: Calculate quantity factor from review count (log normalization)
max_reviews = filtered_df['user_ratings_total'].max()
if max_reviews > 0:
    filtered_df['quantity_factor'] = np.log1p(filtered_df['user_ratings_total']) / np.log1p(max_reviews)
else:
    filtered_df['quantity_factor'] = 1.0

# Step 3: Weighted rating (quality + quantity)
# Formula: weighted_rating = normalized_rating × (1 + quantity_factor) / 2
filtered_df['weighted_rating'] = (
    filtered_df['normalized_google_rating'] * (1 + filtered_df['quantity_factor'])
) / 2

# Step 4: Calculate popularity score (WPI base)
filtered_df['popularity_score'] = (
    0.5 * filtered_df['normalized_google_rating'] + 
    0.5 * filtered_df['weighted_rating']
)

print("📊 Popularity Scoring:")
print(f"   Normalized rating range: {filtered_df['normalized_google_rating'].min():.3f} - {filtered_df['normalized_google_rating'].max():.3f}")
print(f"   Quantity factor range: {filtered_df['quantity_factor'].min():.3f} - {filtered_df['quantity_factor'].max():.3f}")
print(f"   Popularity score range: {filtered_df['popularity_score'].min():.3f} - {filtered_df['popularity_score'].max():.3f}")

# Display top POIs by popularity
print(f"\n⭐ Top 10 POIs by Popularity Score:")
top_pois = filtered_df.nlargest(10, 'popularity_score')[['name', 'rating', 'user_ratings_total', 'popularity_score']]
for idx, row in top_pois.iterrows():
    print(f"   {row['name'][:40]:40s}: {row['popularity_score']:.3f} (⭐{row['rating']}, {int(row['user_ratings_total'])} reviews)")

print(f"\n✅ Module 2 complete: Popularity scores calculated")

## 5. Module 3: Geographic Clustering (K-Means)

Group POIs into days using K-Means clustering (K = number of days).

In [ ]:
# Check if we have enough POIs
if len(filtered_df) < NUM_DAYS:
    raise ValueError(f"Not enough POIs ({len(filtered_df)}) for {NUM_DAYS} days")

# Prepare coordinates
coords = filtered_df[['latitude', 'longitude']].values

# K-Means clustering
kmeans = KMeans(n_clusters=NUM_DAYS, random_state=42, n_init=10)
filtered_df['cluster'] = kmeans.fit_predict(coords)
filtered_df['day'] = filtered_df['cluster'] + 1

print(f"🗺️  K-Means Clustering Results (K={NUM_DAYS}):")
for day in range(1, NUM_DAYS + 1):
    day_pois = filtered_df[filtered_df['day'] == day]
    print(f"   Day {day}: {len(day_pois)} POIs")

print(f"\n✅ Module 3 complete")

In [ ]:
# Normalize popularity within each cluster
for day in range(1, NUM_DAYS + 1):
    day_mask = filtered_df['day'] == day
    day_data = filtered_df[day_mask]
    max_pop = day_data['popularity_score'].max()
    if max_pop > 0:
        filtered_df.loc[day_mask, 'normalized_popularity'] = (
            filtered_df.loc[day_mask, 'popularity_score'] / max_pop
        )
    else:
        filtered_df.loc[day_mask, 'normalized_popularity'] = 0.5

print('✅ Popularity normalized within clusters')

### 5.1 POI Selection Strategy

To optimize GA performance and ensure quality routes, we select top POIs per category:

**Strategy:**
- Top 6 POIs from each category (by normalized_popularity)
- Minimum 15 total POIs per day (ensures route variety)
- Maximum 50 total POIs per day (keeps GA fast)

**Benefits:**
- ✅ Smaller distance matrices (faster OSRM/haversine)
- ✅ Faster GA optimization (fewer combinations)
- ✅ Better route quality (focus on top-rated POIs)
- ✅ Category balance (diverse experiences)

**Example:** If day has 40 POIs across 5 categories:
- Select top 6 from each category = ~30 POIs (some overlap)
- GA optimizes route from these 30 best options
- Final route contains 4-15 POIs (MIN_POIS_PER_ROUTE to MAX_POIS_PER_ROUTE)

In [ ]:
def select_top_pois_per_day(df, day, top_n_per_category=6, min_total=15, max_total=50):
    """
    Select top POIs for a day, balancing across categories.
    
    Strategy:
    1. Get top N POIs from each category (by normalized_popularity)
    2. Ensure minimum total POIs
    3. Cap at maximum to keep GA efficient
    
    Args:
        df: DataFrame with all POIs
        day: Day number to filter
        top_n_per_category: Number of top POIs per category
        min_total: Minimum total POIs to return
        max_total: Maximum total POIs (for GA efficiency)
    
    Returns:
        DataFrame with selected POIs
    """
    day_data = df[df['day'] == day].copy()
    
    if len(day_data) == 0:
        return day_data
    
    # Get unique categories for this day
    all_categories = set()
    for cats in day_data['categories']:
        if isinstance(cats, list):
            all_categories.update(cats)
    
    # Select top N from each category
    selected_pois = set()
    
    for category in all_categories:
        # Filter POIs with this category
        category_pois = day_data[day_data['categories'].apply(
            lambda cats: category in cats if isinstance(cats, list) else False
        )]
        
        # Get top N by normalized popularity
        top_pois = category_pois.nlargest(top_n_per_category, 'normalized_popularity')
        selected_pois.update(top_pois.index)
    
    # Get selected POIs
    result = day_data.loc[list(selected_pois)]
    
    # Ensure minimum total POIs (add highest-rated if needed)
    if len(result) < min_total:
        remaining = day_data[~day_data.index.isin(selected_pois)]
        additional = remaining.nlargest(min_total - len(result), 'normalized_popularity')
        result = pd.concat([result, additional])
    
    # Cap at maximum (keep top by popularity)
    if len(result) > max_total:
        result = result.nlargest(max_total, 'normalized_popularity')
    
    # Sort by normalized_popularity for better display
    result = result.sort_values('normalized_popularity', ascending=False)
    
    return result

print('✅ POI selection function defined')
print(f'   Strategy: Top {6} POIs per category, min {15}, max {50} total')

## 6. Module 4: Genetic Algorithm Route Optimization

### 6.1 POI Class & Distance Functions

In [ ]:
class POI:
    def __init__(self, name, lat, lon, normalized_popularity, place_id=None, rating=4.0, user_ratings_total=0, types=''):
        self.name = name
        self.lat = lat
        self.lon = lon
        self.normalized_popularity = normalized_popularity
        self.place_id = place_id or name
        self.rating = rating
        self.user_ratings_total = user_ratings_total
        self.opening_time = DEFAULT_OPENING_TIME
        self.closing_time = DEFAULT_CLOSING_TIME
        self.visit_duration_min = self._get_visit_duration(types)
    
    def _get_visit_duration(self, types):
        types_lower = str(types).lower()
        for loc_type, duration in VISIT_DURATIONS.items():
            if loc_type != 'default' and loc_type in types_lower:
                return duration
        return VISIT_DURATIONS['default']
    
    def __repr__(self):
        return f"POI({self.name[:30]}, WPI={self.normalized_popularity:.3f})"

def load_pois_for_day(df, day):
    day_data = df[df['day'] == day]
    pois = []
    for _, row in day_data.iterrows():
        poi = POI(
            name=row['name'],
            lat=row['latitude'],
            lon=row['longitude'],
            normalized_popularity=row['normalized_popularity'],
            place_id=row.get('place_id', row['name']),
            rating=row['rating'],
            user_ratings_total=row.get('user_ratings_total', 0),
            types=row.get('types', '')
        )
        pois.append(poi)
    return pois

print('✅ POI class defined')

In [ ]:
def haversine_distance(coord1, coord2):
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return c * 6371

def get_osrm_distance_matrix(pois):
    if not osrm_available:
        return None
    coords_str = ';'.join([f"{poi.lon},{poi.lat}" for poi in pois])
    url = f"{OSRM_BASE_URL}/table/v1/driving/{coords_str}?annotations=distance,duration"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            distance_matrix = np.array(data['distances']) / 1000.0
            duration_matrix = np.array(data['durations']) / 60.0
            return distance_matrix, duration_matrix
    except:
        pass
    return None

def build_distance_matrix(pois):
    global CURRENT_DISTANCE_KM, CURRENT_DURATION_MIN, CURRENT_POI_INDEX
    n = len(pois)
    result = get_osrm_distance_matrix(pois)
    if result:
        CURRENT_DISTANCE_KM, CURRENT_DURATION_MIN = result
        print(f"   ✅ OSRM routing ({n}×{n})")
    else:
        print(f"   ⚠️  Haversine fallback ({n}×{n})")
        CURRENT_DISTANCE_KM = np.zeros((n, n))
        CURRENT_DURATION_MIN = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                if i != j:
                    dist = haversine_distance((pois[i].lat, pois[i].lon), (pois[j].lat, pois[j].lon))
                    CURRENT_DISTANCE_KM[i, j] = dist
                    CURRENT_DURATION_MIN[i, j] = (dist / AVERAGE_SPEED_KM_H) * 60
    CURRENT_POI_INDEX = {poi.place_id: i for i, poi in enumerate(pois)}

def get_cached_distance(poi1, poi2):
    if CURRENT_DISTANCE_KM is None:
        return haversine_distance((poi1.lat, poi1.lon), (poi2.lat, poi2.lon))
    i, j = CURRENT_POI_INDEX.get(poi1.place_id), CURRENT_POI_INDEX.get(poi2.place_id)
    return CURRENT_DISTANCE_KM[i, j] if i is not None and j is not None else haversine_distance((poi1.lat, poi1.lon), (poi2.lat, poi2.lon))

def calculate_travel_time(poi1, poi2):
    if CURRENT_DURATION_MIN is None:
        return (get_cached_distance(poi1, poi2) / AVERAGE_SPEED_KM_H) * 60
    i, j = CURRENT_POI_INDEX.get(poi1.place_id), CURRENT_POI_INDEX.get(poi2.place_id)
    return CURRENT_DURATION_MIN[i, j] if i is not None and j is not None else (get_cached_distance(poi1, poi2) / AVERAGE_SPEED_KM_H) * 60

def time_to_minutes(time_str):
    h, m = map(int, time_str.split(':'))
    return h * 60 + m

def minutes_to_time(minutes):
    return f"{int(minutes//60):02d}:{int(minutes%60):02d}"

print('✅ Distance & time functions defined')

### 6.2 Fitness Function (REWARD + PENALTY Model)

In [ ]:
@dataclass
class RouteEvaluation:
    poi_value_sum: float = 0.0
    distance_penalty: float = 0.0
    user_pref_penalty: float = 0.0
    hard_violation_penalty: float = 0.0
    must_see_penalty: float = 0.0
    restaurant_penalty: float = 0.0
    total_distance_km: float = 0.0
    total_travel_time_min: float = 0.0
    total_visit_time_min: float = 0.0
    closed_poi_count: int = 0
    lunch_invasion_count: int = 0
    overtime_minutes: float = 0.0
    total_time_min: float = 0.0
    timeline: list = field(default_factory=list)
    delta: float = 0.0
    fitness: float = 0.0
    travel_penalty: float = 0.0
    constraint_penalty: float = 0.0

def evaluate_fitness(route, start_time=TOUR_START_TIME):
    eval_result = RouteEvaluation()
    if not route:
        eval_result.delta = float('inf')
        return eval_result
    
    current_time_min = time_to_minutes(start_time)
    tour_start_min = time_to_minutes(TOUR_START_TIME)
    lunch_start_min = time_to_minutes(LUNCH_START_TIME)
    lunch_end_min = time_to_minutes(LUNCH_END_TIME)
    daily_budget_min = DAILY_TIME_BUDGET_HOURS * 60
    
    eval_result.poi_value_sum = sum(poi.normalized_popularity for poi in route)
    
    for i, poi in enumerate(route):
        if i > 0:
            travel_dist_km = get_cached_distance(route[i-1], poi)
            travel_time_min = calculate_travel_time(route[i-1], poi)
            eval_result.distance_penalty += travel_dist_km * DISTANCE_PENALTY_MULTIPLIER
            current_time_min += travel_time_min
            eval_result.total_travel_time_min += travel_time_min
            eval_result.total_distance_km += travel_dist_km
        
        user_preference_pct = poi.normalized_popularity * 100
        eval_result.user_pref_penalty += (100 - user_preference_pct) * USER_PREFERENCE_PENALTY_MULTIPLIER
        
        if lunch_start_min <= current_time_min < lunch_end_min:
            eval_result.lunch_invasion_count += 1
            current_time_min = max(current_time_min, lunch_end_min)
        
        arrival_time = current_time_min
        poi_opening_min = time_to_minutes(poi.opening_time)
        poi_closing_min = time_to_minutes(poi.closing_time)
        
        if current_time_min < poi_opening_min or current_time_min >= poi_closing_min:
            eval_result.closed_poi_count += 1
            eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY
            if current_time_min < poi_opening_min:
                current_time_min = poi_opening_min
        
        visit_start = current_time_min
        visit_end = current_time_min + poi.visit_duration_min
        current_time_min = visit_end
        eval_result.total_visit_time_min += poi.visit_duration_min
        
        if visit_end > poi_closing_min:
            eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY
        
        eval_result.timeline.append((poi, minutes_to_time(int(arrival_time)), minutes_to_time(int(visit_start)), minutes_to_time(int(visit_end))))
    
    eval_result.total_time_min = current_time_min - tour_start_min
    eval_result.overtime_minutes = max(0, eval_result.total_time_min - daily_budget_min)
    
    eval_result.delta = (
        eval_result.distance_penalty +
        eval_result.user_pref_penalty +
        eval_result.hard_violation_penalty +
        eval_result.must_see_penalty +
        eval_result.restaurant_penalty
    )
    
    eval_result.travel_penalty = eval_result.distance_penalty
    eval_result.constraint_penalty = eval_result.hard_violation_penalty + eval_result.must_see_penalty + eval_result.restaurant_penalty
    eval_result.fitness = eval_result.poi_value_sum / (1.0 + eval_result.delta)
    
    return eval_result

print('✅ Fitness function defined')

### 6.3 GA Operators (Selection, Crossover, Mutation)

In [ ]:
class Individual:
    def __init__(self, route, candidate_pois):
        self.route = route.copy() if isinstance(route, list) else list(route)
        self.candidate_pois = candidate_pois
        self.fitness = None
        self.evaluation = None
    
    def evaluate(self, start_time=TOUR_START_TIME):
        self.evaluation = evaluate_fitness(self.route, start_time)
        self.fitness = self.evaluation.fitness
        return self.fitness
    
    def copy(self):
        new_ind = Individual(self.route, self.candidate_pois)
        new_ind.fitness = self.fitness
        new_ind.evaluation = self.evaluation
        return new_ind

def initialize_population(candidate_pois, population_size):
    population = []
    for _ in range(population_size):
        route_len = random.randint(MIN_POIS_PER_ROUTE, min(MAX_POIS_PER_ROUTE, len(candidate_pois)))
        sorted_pois = sorted(candidate_pois, key=lambda p: p.normalized_popularity, reverse=True)
        mid = len(sorted_pois) // 2
        route = []
        for _ in range(route_len):
            if random.random() < 0.7 and mid > 0:
                poi = random.choice(sorted_pois[:mid])
            else:
                poi = random.choice(sorted_pois)
            if poi not in route:
                route.append(poi)
        while len(route) < MIN_POIS_PER_ROUTE:
            unvisited = [p for p in candidate_pois if p not in route]
            if unvisited:
                route.append(random.choice(unvisited))
            else:
                break
        population.append(Individual(route, candidate_pois))
    return population

def tournament_selection(population, tournament_size=TOURNAMENT_SIZE):
    tournament = random.sample(population, tournament_size)
    return max(tournament, key=lambda ind: ind.fitness)

def swap_mutation(individual):
    if random.random() > MUTATION_RATE or len(individual.route) < 2:
        return individual
    mutated = individual.copy()
    idx1, idx2 = random.sample(range(len(mutated.route)), 2)
    mutated.route[idx1], mutated.route[idx2] = mutated.route[idx2], mutated.route[idx1]
    mutated.fitness = None
    return mutated

def single_point_crossover(parent1, parent2):
    min_size = min(len(parent1.route), len(parent2.route))
    if min_size < 2:
        return parent1.copy(), parent2.copy()
    cx_point = random.randint(1, min_size - 1)
    offspring1_route = parent1.route[:cx_point].copy()
    offspring2_route = parent2.route[:cx_point].copy()
    for poi in parent2.route[cx_point:]:
        if poi not in offspring1_route:
            offspring1_route.append(poi)
    for poi in parent1.route[cx_point:]:
        if poi not in offspring2_route:
            offspring2_route.append(poi)
    all_pois = parent1.candidate_pois
    while len(offspring1_route) < MIN_POIS_PER_ROUTE:
        unvisited = [p for p in all_pois if p not in offspring1_route]
        if unvisited:
            offspring1_route.append(random.choice(sorted(unvisited, key=lambda p: p.normalized_popularity, reverse=True)[:5]))
        else:
            break
    while len(offspring2_route) < MIN_POIS_PER_ROUTE:
        unvisited = [p for p in all_pois if p not in offspring2_route]
        if unvisited:
            offspring2_route.append(random.choice(sorted(unvisited, key=lambda p: p.normalized_popularity, reverse=True)[:5]))
        else:
            break
    return Individual(offspring1_route, parent1.candidate_pois), Individual(offspring2_route, parent2.candidate_pois)

print('✅ GA operators defined')

### 6.4 GA Evolution Loop

In [ ]:
def optimize_route_ga(pois, start_time=TOUR_START_TIME, keep_top_n=3):
    """
    Genetic Algorithm with top-N solution tracking.
    
    Args:
        pois: List of POI objects
        start_time: Tour start time
        keep_top_n: Number of top solutions to track (default: 3)
    
    Returns:
        Tuple of (best_individual, top_n_individuals, best_fitness_history)
    """
    if not pois:
        raise ValueError("No POIs provided")
    if len(pois) < MIN_POIS_PER_ROUTE:
        print(f"⚠️  Only {len(pois)} POIs, creating route with all")
        ind = Individual(pois, pois)
        return ind, [ind], [1.0]
    
    print(f"\n🚀 Starting GA with {len(pois)} POIs (tracking top {keep_top_n} solutions)")
    population = initialize_population(pois, POPULATION_SIZE)
    for ind in population:
        ind.evaluate(start_time)
    
    # Track top N solutions (using dict to avoid duplicates)
    top_solutions = {}  # key: route tuple, value: Individual
    
    def update_top_solutions(population):
        for ind in population:
            route_key = tuple(poi.place_id for poi in ind.route)
            if route_key not in top_solutions or ind.fitness > top_solutions[route_key].fitness:
                top_solutions[route_key] = ind.copy()
    
    update_top_solutions(population)
    best_individual = max(population, key=lambda ind: ind.fitness)
    best_fitness_history = [best_individual.fitness]
    print(f"   Gen 0: Best fitness = {best_individual.fitness:.6f}")
    
    no_improvement_count = 0
    for generation in range(1, MAX_GENERATIONS + 1):
        new_population = [best_individual.copy()]
        while len(new_population) < POPULATION_SIZE:
            parent1 = tournament_selection(population)
            parent2 = tournament_selection(population)
            if random.random() < CROSSOVER_RATE:
                offspring1, offspring2 = single_point_crossover(parent1, parent2)
            else:
                offspring1, offspring2 = parent1.copy(), parent2.copy()
            offspring1 = swap_mutation(offspring1)
            offspring2 = swap_mutation(offspring2)
            new_population.extend([offspring1, offspring2])
        
        population = new_population[:POPULATION_SIZE]
        for ind in population:
            if ind.fitness is None:
                ind.evaluate(start_time)
        
        update_top_solutions(population)
        generation_best = max(population, key=lambda ind: ind.fitness)
        if generation_best.fitness > best_individual.fitness:
            best_individual = generation_best.copy()
            no_improvement_count = 0
        else:
            no_improvement_count += 1
        
        best_fitness_history.append(best_individual.fitness)
        
        if generation % 10 == 0:
            print(f"   Gen {generation}: Best fitness = {best_individual.fitness:.6f}, Route length = {len(best_individual.route)}, Top solutions: {len(top_solutions)}")
        
        if no_improvement_count >= EARLY_STOPPING_THRESHOLD:
            print(f"\n⏹️  Early stopping at gen {generation}")
            break
    
    # Get top N unique solutions by fitness
    top_n = sorted(top_solutions.values(), key=lambda x: x.fitness, reverse=True)[:keep_top_n]
    
    # Re-evaluate all top solutions to ensure fresh data
    for ind in top_n:
        ind.evaluate(start_time)
    
    best_individual = top_n[0] if top_n else best_individual
    
    print(f"\n✅ GA completed:")
    print(f"   Final gen: {min(generation, MAX_GENERATIONS)}")
    print(f"   Top {len(top_n)} unique solutions found:")
    for rank, ind in enumerate(top_n, 1):
        print(f"      #{rank}: Fitness={ind.fitness:.6f}, POIs={len(ind.route)}, Distance={ind.evaluation.total_distance_km:.1f}km")
    
    return best_individual, top_n, best_fitness_history

print('✅ GA evolution function with top-N tracking defined')

## 7. Run Optimization for All Days

In [ ]:
# Store results (now with top 3 per day)
all_routes = {}  # best route per day
all_top_routes = {}  # top 3 routes per day
all_fitness_histories = {}

print(f"\n{'='*70}")
print(f"🗓️  OPTIMIZING ROUTES FOR {NUM_DAYS}-DAY TRIP")
print('='*70)

for day in range(1, NUM_DAYS + 1):
    print(f"\n\n📅 DAY {day}")
    print('─' * 70)
    
    # Load and select POIs for this day
    day_df = select_top_pois_per_day(filtered_df, day, top_n_per_category=6, min_total=15, max_total=50)
    print(f"\n✅ Selected {len(day_df)} POIs for Day {day} (from {len(filtered_df[filtered_df['day'] == day])} available)")
    
    # Convert to POI objects
    day_pois = load_pois_for_day(day_df, day)
    
    if len(day_pois) == 0:
        print(f"⚠️  No POIs for Day {day}, skipping")
        continue
    
    # Build distance matrix
    print(f"\n📊 Building distance/time matrices...")
    build_distance_matrix(day_pois)
    
    # Run GA (now returns top 3)
    print(f"\n🧬 Running Genetic Algorithm...")
    best_route, top_3_routes, fitness_history = optimize_route_ga(day_pois, TOUR_START_TIME, keep_top_n=3)
    
    # Store results
    all_routes[day] = best_route
    all_top_routes[day] = top_3_routes
    all_fitness_histories[day] = fitness_history
    
    # Display summary for all 3 alternatives
    print(f"\n📋 Day {day} - Top 3 Route Alternatives:")
    print(f"{'Rank':<6} {'Fitness':<10} {'POIs':<6} {'Distance':<12} {'Time':<10} {'Violations':<12}")
    print('─' * 70)
    for rank, route in enumerate(top_3_routes, 1):
        violations = route.evaluation.closed_poi_count + route.evaluation.lunch_invasion_count
        marker = "⭐ BEST" if rank == 1 else ""
        print(f"#{rank:<5} {route.fitness:<10.6f} {len(route.route):<6} "
              f"{route.evaluation.total_distance_km:<12.2f} "
              f"{route.evaluation.total_time_min/60:<10.2f} "
              f"{violations:<12} {marker}")
    
    print(f"\n✨ Recommended: Route #{1} - {', '.join([poi.name[:20] for poi in top_3_routes[0].route[:3]])}...")

print(f"\n\n{'='*70}")
print(f"✅ ALL DAYS OPTIMIZED (with 3 alternatives each)")
print('='*70)

In [ ]:
# Compare top 3 alternatives for each day
print("\n" + "="*80)
print("🔍 ROUTE ALTERNATIVES COMPARISON")
print("="*80)

for day in sorted(all_top_routes.keys()):
    top_routes = all_top_routes[day]
    
    print(f"\n📅 DAY {day} - Choose Your Route:\n")
    
    for rank, route in enumerate(top_routes, 1):
        eval_result = route.evaluation
        
        print(f"{'─'*80}")
        print(f"Option #{rank} {'⭐ RECOMMENDED' if rank == 1 else ''}")
        print(f"{'─'*80}")
        print(f"Fitness: {route.fitness:.6f} | POIs: {len(route.route)} | Distance: {eval_result.total_distance_km:.1f}km | Time: {eval_result.total_time_min/60:.1f}h")
        print(f"POI Value: {eval_result.poi_value_sum:.3f} | Penalties: {eval_result.delta:.2f}")
        print(f"Violations: {eval_result.closed_poi_count} closed, {eval_result.lunch_invasion_count} lunch invasions, {eval_result.overtime_minutes:.0f}min overtime")
        
        # Show route
        print(f"\nRoute ({len(route.route)} stops):")
        for i, poi in enumerate(route.route, 1):
            arrival = eval_result.timeline[i-1][1] if i-1 < len(eval_result.timeline) else 'N/A'
            print(f"  {i}. {arrival} - {poi.name[:50]}")
        print()
    
    # Recommendation
    best = top_routes[0]
    print(f"\n💡 Recommendation: Option #1")
    if len(top_routes) > 1:
        alt = top_routes[1]
        if best.evaluation.total_distance_km > alt.evaluation.total_distance_km * 1.2:
            print(f"   Note: Option #2 is {(best.evaluation.total_distance_km/alt.evaluation.total_distance_km - 1)*100:.0f}% shorter but slightly lower fitness")
        if len(best.route) < len(alt.route):
            print(f"   Note: Option #2 includes {len(alt.route) - len(best.route)} more POIs")
    print()

print("="*80)
print("✅ Review complete - all alternatives available in all_top_routes dictionary")
print("="*80)

## 8. Visualization & Results

### 8.1 Route Maps (Folium)

In [ ]:
# Color scheme for days
DAY_COLORS = ['blue', 'green', 'red', 'purple', 'orange', 'darkred', 'lightblue']

def create_route_map(day, route_individual):
    route = route_individual.route
    evaluation = route_individual.evaluation
    
    # Center map on route centroid
    center_lat = sum(poi.lat for poi in route) / len(route)
    center_lon = sum(poi.lon for poi in route) / len(route)
    
    # Create map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=11)
    
    # Get day color
    color = DAY_COLORS[(day - 1) % len(DAY_COLORS)]
    
    # Add markers for each POI with timeline info
    for i, (poi, arrival, visit_start, visit_end) in enumerate(evaluation.timeline, 1):
        popup_html = f"""
        <div style='font-family: Arial; width: 200px;'>
            <h4>{i}. {poi.name}</h4>
            <b>Arrival:</b> {arrival}<br>
            <b>Visit:</b> {visit_start} - {visit_end}<br>
            <b>WPI:</b> {poi.normalized_popularity:.3f}<br>
            <b>Rating:</b> {poi.rating}⭐<br>
            <b>Duration:</b> {poi.visit_duration_min} min
        </div>
        """
        
        folium.Marker(
            location=[poi.lat, poi.lon],
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"{i}. {poi.name}",
            icon=folium.Icon(color=color, icon='info-sign', prefix='glyphicon')
        ).add_to(m)
    
    # Draw route polyline
    route_coords = [[poi.lat, poi.lon] for poi in route]
    folium.PolyLine(
        locations=route_coords,
        color=color,
        weight=3,
        opacity=0.7,
        popup=f"Day {day} Route"
    ).add_to(m)
    
    # Add title
    title_html = f'''<h3 style="position: fixed; top: 10px; left: 50px; z-index: 1000; background: white; padding: 10px; border-radius: 5px;">
    Day {day} Route - {len(route)} POIs
    </h3>'''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m

print('✅ Map creation function defined')

In [ ]:
# Create and display maps for all days (best route + optionally alternatives)
print("\n🗺️  Creating route maps...\n")

SHOW_ALTERNATIVES = False  # Set to True to generate maps for all 3 alternatives

for day in sorted(all_routes.keys()):
    # Always create map for best route
    route_map = create_route_map(day, all_routes[day])
    
    # Save to file
    map_filename = f'wanderwise_route_day{day}_map.html'
    route_map.save(map_filename)
    print(f"✅ Day {day} (Best) map saved: {map_filename}")
    
    # Optionally create maps for alternatives
    if SHOW_ALTERNATIVES and day in all_top_routes:
        for rank, route in enumerate(all_top_routes[day][1:], 2):  # Skip first (already saved)
            alt_map = create_route_map(day, route)
            alt_filename = f'wanderwise_route_day{day}_alternative{rank}_map.html'
            alt_map.save(alt_filename)
            print(f"   Alternative #{rank}: {alt_filename}")
    
    # Display best route in notebook
    print(f"\n📍 Day {day} Map (Best Route):")
    display(route_map)

if not SHOW_ALTERNATIVES:
    print(f"\n💡 Tip: Set SHOW_ALTERNATIVES=True to generate maps for all 3 route alternatives")

print("\n✅ All route maps created")

### 8.2 Timeline Tables

In [ ]:
# Display timeline for each day
for day in sorted(all_routes.keys()):
    route_ind = all_routes[day]
    evaluation = route_ind.evaluation
    
    print(f"\n{'='*80}")
    print(f"📅 DAY {day} TIMELINE - {len(route_ind.route)} POIs")
    print('='*80)
    print(f"Start: {TOUR_START_TIME} | Total time: {evaluation.total_time_min/60:.2f}h | Distance: {evaluation.total_distance_km:.2f}km")
    print(f"Fitness: {route_ind.fitness:.6f} | Violations: {evaluation.closed_poi_count} closed, {evaluation.lunch_invasion_count} lunch invasions\n")
    
    print(f"{'#':<3} {'Arrival':<8} {'POI Name':<35} {'Visit Window':<15} {'Duration':<10}")
    print('─' * 80)
    
    for i, (poi, arrival, visit_start, visit_end) in enumerate(evaluation.timeline, 1):
        # Check for violations
        marker = ''
        if time_to_minutes(arrival) < time_to_minutes(poi.opening_time):
            marker = '⚠️ EARLY'
        elif time_to_minutes(arrival) >= time_to_minutes(poi.closing_time):
            marker = '❌ CLOSED'
        
        print(f"{i:<3} {arrival:<8} {poi.name[:35]:<35} {visit_start}-{visit_end:<8} {poi.visit_duration_min} min {marker}")
    
    print('─' * 80)
    print(f"End time: {evaluation.timeline[-1][3] if evaluation.timeline else 'N/A'}")
    print(f"Overtime: {evaluation.overtime_minutes:.0f} min" if evaluation.overtime_minutes > 0 else "✅ Within time budget")

### 8.3 GA Convergence Charts (Plotly)

In [ ]:
if PLOTLY_AVAILABLE:
    # Create convergence chart for all days
    fig = go.Figure()
    
    for day in sorted(all_fitness_histories.keys()):
        fitness_history = all_fitness_histories[day]
        generations = list(range(len(fitness_history)))
        
        fig.add_trace(go.Scatter(
            x=generations,
            y=fitness_history,
            mode='lines+markers',
            name=f'Day {day}',
            line=dict(width=2)
        ))
    
    fig.update_layout(
        title='Genetic Algorithm Fitness Convergence',
        xaxis_title='Generation',
        yaxis_title='Fitness Score',
        hovermode='x unified',
        template='plotly_white',
        height=500
    )
    
    fig.show()
    print("\n✅ Convergence chart displayed")
else:
    print("⚠️  Plotly not available, skipping convergence chart")

In [ ]:
# Summary statistics
print("\n📊 TRIP SUMMARY STATISTICS")
print('='*70)

total_pois = sum(len(all_routes[day].route) for day in all_routes)
total_distance = sum(all_routes[day].evaluation.total_distance_km for day in all_routes)
total_time = sum(all_routes[day].evaluation.total_time_min for day in all_routes)
avg_fitness = sum(all_routes[day].fitness for day in all_routes) / len(all_routes) if all_routes else 0

print(f"\n🗓️  Trip Duration: {NUM_DAYS} days")
print(f"📍 Total POIs: {total_pois}")
print(f"🚗 Total Distance: {total_distance:.2f} km")
print(f"⏱️  Total Time: {total_time/60:.2f} hours ({total_time/60/NUM_DAYS:.2f} hours/day avg)")
print(f"🎯 Average Fitness: {avg_fitness:.6f}")

print(f"\n📋 Per-Day Breakdown:")
for day in sorted(all_routes.keys()):
    route = all_routes[day]
    print(f"   Day {day}: {len(route.route)} POIs, {route.evaluation.total_distance_km:.1f}km, "
          f"{route.evaluation.total_time_min/60:.2f}h, fitness={route.fitness:.4f}")

print(f"\n✅ Optimization complete!")

## 9. Export Results

### 9.1 Export to JSON

In [ ]:
# Build JSON export structure with top 3 alternatives per day
export_data = {
    'trip_config': {
        'num_days': NUM_DAYS,
        'min_rating': MIN_RATING,
        'min_pois_per_route': MIN_POIS_PER_ROUTE,
        'max_pois_per_route': MAX_POIS_PER_ROUTE,
        'user_preferences': USER_PREFERENCE_INPUT.strip()
    },
    'ga_config': {
        'population_size': POPULATION_SIZE,
        'max_generations': MAX_GENERATIONS,
        'crossover_rate': CROSSOVER_RATE,
        'mutation_rate': MUTATION_RATE,
        'tournament_size': TOURNAMENT_SIZE
    },
    'routes': {}
}

# Add route data with alternatives
for day in sorted(all_top_routes.keys()):
    top_routes = all_top_routes[day]
    
    day_alternatives = []
    
    for rank, route_ind in enumerate(top_routes, 1):
        eval_result = route_ind.evaluation
        
        route_data = {
            'rank': rank,
            'recommended': (rank == 1),
            'pois': [],
            'timeline': [],
            'statistics': {
                'total_pois': len(route_ind.route),
                'total_distance_km': round(eval_result.total_distance_km, 2),
                'total_time_hours': round(eval_result.total_time_min / 60, 2),
                'fitness': round(route_ind.fitness, 6),
                'poi_value_sum': round(eval_result.poi_value_sum, 3),
                'delta_penalty': round(eval_result.delta, 2),
                'violations': {
                    'closed_pois': eval_result.closed_poi_count,
                    'lunch_invasions': eval_result.lunch_invasion_count,
                    'overtime_minutes': round(eval_result.overtime_minutes, 1)
                }
            }
        }
        
        # Add POI details
        for poi in route_ind.route:
            route_data['pois'].append({
                'name': poi.name,
                'place_id': poi.place_id,
                'latitude': poi.lat,
                'longitude': poi.lon,
                'normalized_popularity': round(poi.normalized_popularity, 3),
                'rating': poi.rating,
                'visit_duration_min': poi.visit_duration_min
            })
        
        # Add timeline
        for poi, arrival, visit_start, visit_end in eval_result.timeline:
            route_data['timeline'].append({
                'poi_name': poi.name,
                'arrival_time': arrival,
                'visit_start': visit_start,
                'visit_end': visit_end
            })
        
        day_alternatives.append(route_data)
    
    export_data['routes'][f'day_{day}'] = {
        'day': day,
        'alternatives': day_alternatives,
        'recommended_route': day_alternatives[0] if day_alternatives else None
    }

# Save to JSON
json_filename = 'wanderwise_optimized_routes.json'
with open(json_filename, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"✅ Routes exported to {json_filename}")
print(f"   File size: {len(json.dumps(export_data))} bytes")
print(f"   {NUM_DAYS} days × 3 alternatives = {NUM_DAYS * 3} total routes exported")
print(f"\n📊 Export Summary:")
for day in sorted(all_top_routes.keys()):
    num_alts = len(all_top_routes[day])
    print(f"   Day {day}: {num_alts} route alternatives")

### 9.2 Summary CSV (Optional)

In [ ]:
# Create summary DataFrame
summary_data = []
for day in sorted(all_routes.keys()):
    route_ind = all_routes[day]
    eval_result = route_ind.evaluation
    
    for i, poi in enumerate(route_ind.route, 1):
        summary_data.append({
            'day': day,
            'sequence': i,
            'poi_name': poi.name,
            'latitude': poi.lat,
            'longitude': poi.lon,
            'normalized_popularity': poi.normalized_popularity,
            'rating': poi.rating,
            'visit_duration_min': poi.visit_duration_min
        })

summary_df = pd.DataFrame(summary_data)
csv_filename = 'wanderwise_route_summary.csv'
summary_df.to_csv(csv_filename, index=False)

print(f"✅ Summary CSV exported to {csv_filename}")
print(f"   {len(summary_df)} POI visits across {NUM_DAYS} days")
print(f"\nFirst few rows:")
summary_df.head(10)

## 10. Usage Examples & Tips

### Example Preference Inputs

**Example 1: Beach & Nature Lover**
```python
USER_PREFERENCE_INPUT = """  
I love relaxing on beautiful beaches and exploring waterfalls.
Interested in nature and wildlife sanctuaries.
Not interested in shopping or nightlife.
"""
```

**Example 2: History Buff**
```python
USER_PREFERENCE_INPUT = """  
I'm fascinated by historical forts, churches, and museums.
Love learning about Portuguese colonial heritage.
Not interested in beaches or adventure activities.
"""
```

**Example 3: Adventure Seeker**
```python
USER_PREFERENCE_INPUT = """  
Looking for adventure! Forts, trekking, water sports, kayaking.
Also interested in waterfalls and nature trails.
No interest in religious sites or shopping.
"""
```

**Example 4: Balanced Tourist**
```python
USER_PREFERENCE_INPUT = """  
I enjoy a mix of beaches, historical sites, and local food.
Interested in experiencing Goan culture and religious sites.
"""
```

### Troubleshooting

**Issue: "No places match your preferences"**
- Solution: Lower `MIN_RATING` or use fewer negative filters
- Try broader category keywords in your input

**Issue: "Not enough POIs for N days"**
- Solution: Reduce `NUM_DAYS` or lower `MIN_RATING`
- Use more inclusive preference input

**Issue: GA produces short routes (< MIN_POIS_PER_ROUTE)**
- Check: Distance penalties might be too high
- Check: Time budget might be too restrictive
- Solution: Adjust penalty multipliers or increase daily time budget

**Issue: OSRM not available**
- Solution: Use haversine fallback (automatic)
- For best results: Set up OSRM Docker (see setup instructions)

**Issue: Routes have many constraint violations**
- Check: POI opening/closing times (defaults: 09:00-18:00)
- Adjust: `TOUR_START_TIME`, `TOUR_END_TIME` to match POI schedules
- Note: Hard violation penalty is 1000 per violation

### Parameter Tuning Guide

**To get longer routes:**
- Increase `MAX_POIS_PER_ROUTE`
- Decrease `DISTANCE_PENALTY_MULTIPLIER`
- Increase `DAILY_TIME_BUDGET_HOURS`

**To improve route quality:**
- Increase `POPULATION_SIZE` (slower but better results)
- Increase `MAX_GENERATIONS`
- Adjust `CROSSOVER_RATE` and `MUTATION_RATE`

**To speed up optimization:**
- Decrease `POPULATION_SIZE`
- Decrease `MAX_GENERATIONS`
- Use smaller `NUM_DAYS`

## 🎉 Notebook Complete!

You've successfully created optimized multi-day tourism routes for Goa using:
- ✅ Natural language preference filtering
- ✅ Popularity-based POI scoring (WPI)
- ✅ Geographic clustering (K-Means)
- ✅ Genetic algorithm route optimization (TPOS-aligned)
- ✅ Interactive maps and visualizations

### Next Steps

1. **Try different preferences** - Modify `USER_PREFERENCE_INPUT` and re-run
2. **Adjust trip duration** - Change `NUM_DAYS` for shorter/longer trips
3. **Tune GA parameters** - Experiment with population size, mutation rate, etc.
4. **Export results** - Use the JSON/CSV exports for your application
5. **Visualize** - Open the generated HTML maps in your browser

### Files Generated

- `wanderwise_route_day{N}_map.html` - Interactive maps for each day
- `wanderwise_optimized_routes.json` - Complete route data
- `wanderwise_route_summary.csv` - POI visit schedule

---

**WanderWise+ Architecture References:**
- Module 1: NLC simulation (keyword-based classification)
- Module 2: WPI calculation (Google rating + quantity factor)
- Module 3: K-Means clustering (geographic grouping)
- Module 4: GA optimization (TPOS paper by Rusu & Alexandrescu, 2024)

**For questions or issues:** Check the troubleshooting section above!